# BLIP Feature Extraction + Children's Story Generation
**Pipeline:** Image → BLIP features → Groq Llama 3 → Story

| Step | Description |
|---|---|
| 1-3 | Install, load BLIP, encode prompts |
| 4-5 | Configure paths, batch extract 8,091 images |
| 6 | Save features |
| 7 | Setup Groq LLM |
| 8 | Generate stories for all images |
| 9 | Preview + save stories |

In [10]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers", "torch", "torchvision", "Pillow", "tqdm", "groq"])
print("Dependencies OK")

Dependencies OK


In [11]:
import torch, os, pickle, json, math, time
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
blip_model = BlipModel.from_pretrained("Salesforce/blip-itm-base-coco").to(DEVICE)
blip_model.eval()
print("BLIP loaded")

Device: cpu


[transformers] `BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Loading weights: 100%|██████████| 150/150 [00:00<00:00, 36041.80it/s]
[transformers] BlipModel LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                                                        | Status     | 
---------------------------------------------------------------------------+------------+-
text_encoder.encoder.layer.{0...11}.attention.self.value.bias              | UNEXPECTED | 
text_encoder.encoder.layer.{0...11}.intermediate.dense.bias                | UNEXPECTED | 
text_encoder.encoder.layer.{0...11}.attention.self.key.bias                | UNEXPECTED | 
text_encoder.encoder.layer.{0...11}.crossattention.self.key.weight         | UNEXPECTED | 
text_encoder.encoder.layer.{0...11}.crossattention.self.value.weight       | UNEXPECTED | 
text_encoder.encoder

BLIP loaded


In [12]:
SCENE_PROMPTS = [
    "a magical forest", "a sunny beach", "a cozy village", "a snowy mountain",
    "a busy marketplace", "a quiet farm", "a hidden cave", "a royal palace",
    "a starry night sky", "a flowing river", "a playground with children",
    "a school classroom", "a flower garden", "an adventure in the jungle"
]
OBJECT_PROMPTS = [
    "a friendly dog", "a talking cat", "a wise old owl", "a playful rabbit",
    "a brave lion", "a colorful bird", "a treasure chest", "a magic wand",
    "a knight in armor", "a flying dragon", "a wooden boat", "a golden crown"
]
EMOTION_PROMPTS = [
    "a happy and joyful moment", "a sad and lonely feeling", "an exciting adventure",
    "a peaceful and calm atmosphere", "a curious and wondering expression",
    "a brave and courageous act", "a loving and warm embrace", "a funny and silly situation"
]
SETTING_PROMPTS = [
    "a fantasy fairytale world", "a realistic modern day setting",
    "an ancient historical kingdom", "an underwater ocean world",
    "a cozy indoor home scene", "a wild outdoor nature scene", "a magical enchanted land"
]

ALL_PROMPTS = {
    "top_scenes":   SCENE_PROMPTS,
    "top_objects":  OBJECT_PROMPTS,
    "top_emotions": EMOTION_PROMPTS,
    "top_settings": SETTING_PROMPTS,
}

ENCODED = {}
with torch.no_grad():
    for cat, prompts in ALL_PROMPTS.items():
        inputs = blip_processor(text=prompts, return_tensors="pt", padding=True).to(DEVICE)
        outputs = blip_model.get_text_features(**inputs)
        # Extract pooled features (or [CLS] token)
        if outputs.pooler_output is not None:
            f = outputs.pooler_output.float()
        else:
            f = outputs.last_hidden_state[:, 0, :].float()
        ENCODED[cat] = {"labels": prompts, "features": f / f.norm(dim=-1, keepdim=True)}

In [13]:
IMAGES_DIR  = r"C:/Users/GIGABYTE/Downloads/ImageCaptioningproject-CNN-LSTM-main (2)/Images"
BASE_DIR    = r"C:/Users/GIGABYTE/Downloads/ImageCaptioningproject-CNN-LSTM-main (2)/ImageCaptioningproject-CNN-LSTM-main"
OUTPUT_PKL  = os.path.join(BASE_DIR, "image_features_for_stories.pkl")
OUTPUT_JSON = os.path.join(BASE_DIR, "image_features_for_stories_sample.json")
STORIES_PKL = os.path.join(BASE_DIR, "image_stories.pkl")
STORIES_JSON= os.path.join(BASE_DIR, "image_stories.json")

BATCH_SIZE = 32
TOP_K      = 5
SAVE_EVERY = 500
RESUME     = True
SUPPORTED  = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

image_paths = sorted([
    os.path.join(IMAGES_DIR, f)
    for f in os.listdir(IMAGES_DIR)
    if os.path.splitext(f)[1].lower() in SUPPORTED
])

all_features = []
if RESUME and os.path.exists(OUTPUT_PKL):
    with open(OUTPUT_PKL, "rb") as _f:
        all_features = pickle.load(_f)
    print(f"Resuming: {len(all_features)} already done.")

done_paths = {d["image_path"] for d in all_features}
todo_paths = [p for p in image_paths if p not in done_paths]
print(f"Total: {len(image_paths):,}  |  Done: {len(done_paths):,}  |  Todo: {len(todo_paths):,}")

Total: 8,091  |  Done: 0  |  Todo: 8,091


In [14]:
errors = []

def process_batch(paths):
    valid_images, valid = [], []
    for p in paths:
        try:
            valid_images.append(Image.open(p).convert("RGB"))
            valid.append(p)
        except Exception as e:
            errors.append({"path": p, "error": str(e)})
    if not valid_images:
        return []
    inputs = blip_processor(images=valid_images, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        feats = blip_model.get_image_features(**inputs).float()
    norms = feats / feats.norm(dim=-1, keepdim=True)
    results = []
    for i, path in enumerate(valid):
        vec = norms[i]
        feat = {"image_path": path, "embedding": vec.cpu().numpy().tolist()}
        for cat, data in ENCODED.items():
            sims = (vec @ data["features"].T).cpu().numpy()
            top_idx = np.argsort(sims)[::-1][:TOP_K]
            feat[cat] = [{"label": data["labels"][j], "score": float(sims[j])} for j in top_idx]
        results.append(feat)
    return results

num_batches = math.ceil(len(todo_paths) / BATCH_SIZE)
t0 = time.time()
processed = 0

for b in tqdm(range(num_batches), desc=f"Extracting (batch={BATCH_SIZE})"):
    batch = todo_paths[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
    all_features.extend(process_batch(batch))
    processed += len(batch)
    if processed % SAVE_EVERY < BATCH_SIZE:
        with open(OUTPUT_PKL, "wb") as _f:
            pickle.dump(all_features, _f)
        print(f"  Checkpoint: {len(all_features):,} saved")

print(f"Done! {len(all_features):,} images in {time.time()-t0:.0f}s")

Extracting (batch=32):   0%|          | 0/253 [00:23<?, ?it/s]


AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'float'

In [ ]:
with open(OUTPUT_PKL, "wb") as f:
    pickle.dump(all_features, f)
print(f"Features saved -> {OUTPUT_PKL}")

sample = [{k: v for k, v in feat.items() if k != "embedding"}
          for feat in all_features[:10]]
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(sample, f, indent=2, ensure_ascii=False)
print(f"Sample JSON saved -> {OUTPUT_JSON}")

---
## Story Generation with Groq Llama 3
Uses the BLIP feature bundle from each image to generate a children's story via Groq API.

In [ ]:
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    raise ValueError("Please set GROQ_API_KEY in your environment or Colab Secrets.")

GROQ_MODEL   = "llama3-70b-8192"   # fast, high-quality Llama 3 70B
groq_client  = Groq(api_key=GROQ_API_KEY)
print(f"Groq client ready | model: {GROQ_MODEL}")


In [ ]:
SYSTEM_PROMPT = """You are a creative children's story writer. 
You write short, imaginative, age-appropriate stories for children aged 4-8.
Each story must:
- Be 4-6 sentences long
- Have a clear beginning, middle and happy ending
- Use simple vocabulary
- Include a moral or lesson
- Be engaging, fun and full of wonder
Return ONLY the story text, no titles or extra commentary."""


def build_prompt(feat: dict) -> str:
    scene   = feat["top_scenes"][0]["label"]
    setting = feat["top_settings"][0]["label"]
    mood    = feat["top_emotions"][0]["label"]
    objects = ", ".join(d["label"] for d in feat["top_objects"][:3])
    themes  = ", ".join(d["label"] for d in feat["top_scenes"][:3])
    return (
        f"Write a children's story inspired by this image description:\n"
        f"  Location  : {setting}, specifically {scene}\n"
        f"  Characters: {objects}\n"
        f"  Mood      : {mood}\n"
        f"  Themes    : {themes}\n"
    )


def generate_story(feat: dict, temperature: float = 0.85) -> str:
    """Call Groq Llama 3 and return the generated story string."""
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_prompt(feat)},
        ],
        temperature=temperature,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


# Quick test on the first image
test_story = generate_story(all_features[0])
print("=== Test Story ===")
print(test_story)

In [ ]:
# ── Batch story generation ────────────────────────────────────────────────────
# Change MAX_STORIES to len(all_features) to process all 8,091 images
# Keep it small first to test (Groq free tier: ~30 req/min)
MAX_STORIES   = 50          # set to len(all_features) for full run
STORY_DELAY   = 2.1         # seconds between requests (stays under rate limit)
RESUME_STORIES= True        # skip images that already have a story

# Load existing stories if resuming
all_stories = []
if RESUME_STORIES and os.path.exists(STORIES_PKL):
    with open(STORIES_PKL, "rb") as _f:
        all_stories = pickle.load(_f)
    print(f"Resuming: {len(all_stories)} stories already generated.")

done_story_paths = {s["image_path"] for s in all_stories}
todo_feats = [f for f in all_features[:MAX_STORIES]
              if f["image_path"] not in done_story_paths]
print(f"Generating {len(todo_feats)} new stories...")

story_errors = []

for feat in tqdm(todo_feats, desc="Generating stories"):
    try:
        story = generate_story(feat)
        all_stories.append({
            "image_path": feat["image_path"],
            "image_name": os.path.basename(feat["image_path"]),
            "top_scene":  feat["top_scenes"][0]["label"],
            "top_object": feat["top_objects"][0]["label"],
            "mood":       feat["top_emotions"][0]["label"],
            "setting":    feat["top_settings"][0]["label"],
            "story":      story,
        })
        time.sleep(STORY_DELAY)  # respect rate limit
    except Exception as e:
        story_errors.append({"path": feat["image_path"], "error": str(e)})
        time.sleep(5)            # back off on error

print(f"\nDone! {len(all_stories)} total stories  |  {len(story_errors)} errors")

In [ ]:
# Save stories
with open(STORIES_PKL, "wb") as f:
    pickle.dump(all_stories, f)
print(f"Stories (pkl) saved -> {STORIES_PKL}")

with open(STORIES_JSON, "w", encoding="utf-8") as f:
    json.dump(all_stories, f, indent=2, ensure_ascii=False)
print(f"Stories (json) saved -> {STORIES_JSON}")

# Pretty print the first 3 stories
for s in all_stories[:3]:
    print(f"\n{'='*60}")
    print(f"Image  : {s['image_name']}")
    print(f"Scene  : {s['top_scene']}")
    print(f"Object : {s['top_object']}")
    print(f"Mood   : {s['mood']}")
    print(f"\nSTORY:")
    print(s['story'])

## Single Image: Feature Extraction + Story in One Shot

In [ ]:
import matplotlib.pyplot as plt

def story_from_image(image_path: str, temperature: float = 0.9) -> dict:
    """Full pipeline: image file -> BLIP features -> Groq story."""
    # Extract features
    img = Image.open(image_path).convert("RGB")
    inputs = blip_processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        vec = blip_model.get_image_features(**inputs).float()
    vec = vec / vec.norm(dim=-1, keepdim=True)

    feat = {"image_path": image_path, "embedding": vec.cpu().numpy()[0].tolist()}
    for cat, data in ENCODED.items():
        sims    = (vec[0] @ data["features"].T).cpu().numpy()
        top_idx = np.argsort(sims)[::-1][:TOP_K]
        feat[cat] = [{"label": data["labels"][j], "score": float(sims[j])} for j in top_idx]

    story = generate_story(feat, temperature=temperature)

    return {
        "image_path": image_path,
        "features": feat,
        "story": story,
    }


# ── Demo: pick any image from the dataset ────────────────────────────────────
demo_path = image_paths[42]   # change index to pick a different image

result = story_from_image(demo_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(Image.open(demo_path).convert("RGB"))
axes[0].axis("off")
axes[0].set_title(os.path.basename(demo_path), fontsize=9)

labels = [d["label"] for d in result["features"]["top_scenes"]]
scores = [d["score"] for d in result["features"]["top_scenes"]]
axes[1].barh(range(len(labels)), scores, color="steelblue")
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel("BLIP Cosine Similarity")
axes[1].set_title("Top Scene Matches")
plt.tight_layout()
plt.show()

print(f"\nScene   : {result['features']['top_scenes'][0]['label']}")
print(f"Object  : {result['features']['top_objects'][0]['label']}")
print(f"Mood    : {result['features']['top_emotions'][0]['label']}")
print(f"Setting : {result['features']['top_settings'][0]['label']}")
print(f"\n{'='*50}")
print("GENERATED STORY:")
print(f"{'='*50}")
print(result["story"])